# Pilot V3 - Published Greek Telecom Tower Case

This notebook is different from V1 and V2.

V1 and V2 are pilot surrogate models. This notebook uses published fragility parameters from Bilionis and Vamvatsikos (2022) for a 48 m Greek steel lattice telecommunication tower.

Kid version: instead of inventing the tower strength, we copy the fragility numbers from a real paper and draw the curves so we can compare our future work to something published.

## 1. Imports and Paths

This notebook reads no external files. The published values are written directly in the notebook so the case is transparent and easy to audit.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
from scipy.stats import norm

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

CACHE_DIR = REPO_ROOT / 'outputs' / '_cache'
MPL_CACHE_DIR = REPO_ROOT / 'outputs' / '_matplotlib_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('XDG_CACHE_HOME', str(CACHE_DIR))
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 130
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

OUTPUT_DIR = REPO_ROOT / 'outputs' / 'notebook_v3'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repository root: {REPO_ROOT}')
print(f'Notebook outputs: {OUTPUT_DIR}')

## 2. Published Case Metadata

The paper studies a steel lattice telecommunication tower with a 48 m structural height and dish antennas near the top. The fragility curves are lognormal and are provided for multiple wind directions and tower conditions.

In [ ]:
literature_case = {
    'source_title': 'Risk assessment of rehabilitation strategies for steel lattice telecommunication towers of Greece under extreme wind hazard',
    'authors': ['Dimitrios V. Bilionis', 'Dimitrios I. Vamvatsikos'],
    'year': 2022,
    'journal': 'Engineering Structures',
    'doi': '10.1016/j.engstruct.2022.114625',
    'tower_type': 'steel lattice telecommunication tower',
    'structural_height_m': 48.0,
    'dish_antennas_count': 4,
    'wind_intensity_measure': '10-minute average wind speed',
    'basic_wind_speeds_mps': {
        'most_of_Greece': 27.0,
        'within_10km_of_shoreline': 33.0,
    },
}

literature_case

## 3. Published Fragility Parameters

`u50` is the wind speed where the collapse probability is 50 percent. `beta` controls the spread of the curve. These values come from the published paper's fragility table.

In [ ]:
published_fragility_parameters = [
    {'tower_condition': 'initial_tower', 'direction_deg': 0.0, 'u50_mps': 39.11, 'beta': 0.1895},
    {'tower_condition': 'initial_tower', 'direction_deg': 22.5, 'u50_mps': 42.83, 'beta': 0.1898},
    {'tower_condition': 'initial_tower', 'direction_deg': 45.0, 'u50_mps': 45.94, 'beta': 0.1908},
    {'tower_condition': 'corroded_tower', 'direction_deg': 0.0, 'u50_mps': 30.82, 'beta': 0.1994},
    {'tower_condition': 'corroded_tower', 'direction_deg': 22.5, 'u50_mps': 33.44, 'beta': 0.1985},
    {'tower_condition': 'corroded_tower', 'direction_deg': 45.0, 'u50_mps': 35.93, 'beta': 0.1982},
    {'tower_condition': 'strengthened_tower', 'direction_deg': 0.0, 'u50_mps': 46.03, 'beta': 0.1637},
    {'tower_condition': 'strengthened_tower', 'direction_deg': 22.5, 'u50_mps': 47.89, 'beta': 0.1625},
    {'tower_condition': 'strengthened_tower', 'direction_deg': 45.0, 'u50_mps': 49.27, 'beta': 0.1631},
    {'tower_condition': 'strengthened_tower_with_HSS_bracings', 'direction_deg': 0.0, 'u50_mps': 57.96, 'beta': 0.1379},
    {'tower_condition': 'strengthened_tower_with_HSS_bracings', 'direction_deg': 22.5, 'u50_mps': 58.18, 'beta': 0.1375},
    {'tower_condition': 'strengthened_tower_with_HSS_bracings', 'direction_deg': 45.0, 'u50_mps': 57.13, 'beta': 0.1340},
]

fragility_parameters_df = pd.DataFrame(published_fragility_parameters)
fragility_parameters_df

## 4. Compute Fragility Curves

The published lognormal form is:

`P(collapse | u) = Phi(ln(u / u50) / beta)`

This cell calculates the collapse probability across a smooth wind-speed grid.

In [ ]:
def lognormal_fragility_probability(wind_speed_mps, u50_mps, beta):
    '''Compute collapse probability using the published lognormal fragility form.'''
    wind_speed_mps = np.asarray(wind_speed_mps, dtype=float)
    return norm.cdf(np.log(wind_speed_mps / u50_mps) / beta)


wind_speed_grid_mps = np.arange(20.0, 60.0 + 0.5, 0.5)
curve_records = []

for _, row in fragility_parameters_df.iterrows():
    probabilities = lognormal_fragility_probability(wind_speed_grid_mps, row['u50_mps'], row['beta'])
    for wind_speed_mps, probability in zip(wind_speed_grid_mps, probabilities):
        curve_records.append({
            'tower_condition': row['tower_condition'],
            'direction_deg': float(row['direction_deg']),
            'wind_speed_mps': float(wind_speed_mps),
            'u50_mps': float(row['u50_mps']),
            'beta': float(row['beta']),
            'collapse_probability': float(probability),
        })

fragility_points_df = pd.DataFrame(curve_records)
fragility_points_df.head()

## 5. Plot Published Fragility Curves

Each subplot is a tower condition. The curves show how collapse probability changes with wind speed and direction.

In [ ]:
condition_titles = {
    'initial_tower': 'Initial Tower',
    'corroded_tower': 'Corroded Tower',
    'strengthened_tower': 'Strengthened Tower',
    'strengthened_tower_with_HSS_bracings': 'Strengthened + HSS Bracings',
}
direction_colors = {0.0: 'tab:blue', 22.5: 'tab:orange', 45.0: 'tab:red'}

fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey=True)
axes = axes.flatten()

for axis, tower_condition in zip(axes, condition_titles):
    condition_df = fragility_points_df[fragility_points_df['tower_condition'] == tower_condition]

    for direction_deg in sorted(condition_df['direction_deg'].unique()):
        direction_df = condition_df[condition_df['direction_deg'] == direction_deg]
        axis.plot(
            direction_df['wind_speed_mps'],
            direction_df['collapse_probability'],
            color=direction_colors[direction_deg],
            linewidth=2.2,
            label=f'{direction_deg:g} deg',
        )

    axis.set_title(condition_titles[tower_condition])
    axis.set_xlabel('10-minute average wind speed (m/s)')
    axis.set_ylabel('Probability of collapse')
    axis.set_ylim(-0.02, 1.02)
    axis.legend(fontsize=8)

fig.suptitle('Published fragility curves for a 48 m Greek telecom tower case', fontsize=12)
fig.tight_layout()
plt.show()

## 6. Collapse Probability at Greek Basic Wind Speeds

The paper reports basic wind speeds of 27 m/s for most of Greece and 33 m/s near the shoreline. This table shows the collapse probability at those two wind speeds.

In [ ]:
design_speed_records = []

for _, row in fragility_parameters_df.iterrows():
    for design_case, design_speed_mps in literature_case['basic_wind_speeds_mps'].items():
        probability = lognormal_fragility_probability(design_speed_mps, row['u50_mps'], row['beta'])
        design_speed_records.append({
            'tower_condition': row['tower_condition'],
            'direction_deg': float(row['direction_deg']),
            'design_case': design_case,
            'design_speed_mps': float(design_speed_mps),
            'collapse_probability': float(probability),
        })

design_speed_df = pd.DataFrame(design_speed_records)
design_speed_df.head(12)

## 7. Plot Design-Speed Probabilities

This plot makes the big idea easy to see: corroded towers have much higher collapse probability, and strengthening reduces risk.

In [ ]:
summary_df = (
    design_speed_df.assign(
        condition_short=lambda df: df['tower_condition'].map({
            'initial_tower': 'Initial',
            'corroded_tower': 'Corroded',
            'strengthened_tower': 'Strengthened',
            'strengthened_tower_with_HSS_bracings': 'Strengthened + HSS',
        }),
        design_case_short=lambda df: df['design_case'].map({
            'most_of_Greece': '27 m/s',
            'within_10km_of_shoreline': '33 m/s',
        }),
    )
    .pivot_table(index=['condition_short', 'direction_deg'], columns='design_case_short', values='collapse_probability')
    .reset_index()
)

x = np.arange(len(summary_df))
bar_width = 0.38

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.bar(x - 0.5 * bar_width, summary_df['27 m/s'], width=bar_width, label='27 m/s', color='tab:blue')
ax.bar(x + 0.5 * bar_width, summary_df['33 m/s'], width=bar_width, label='33 m/s', color='tab:red')
ax.set_xticks(x)
ax.set_xticklabels([f"{row['condition_short']}\n{row['direction_deg']:g} deg" for _, row in summary_df.iterrows()], fontsize=8)
ax.set_ylabel('Probability of collapse')
ax.set_title('Published collapse probabilities at Greek basic wind speeds')
ax.set_ylim(0.0, 1.0)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.show()

## 8. Save Notebook Outputs

This saves the literature metadata, published fragility parameters, computed curve points, and design-speed probabilities.

In [ ]:
with (OUTPUT_DIR / 'literature_case_metadata.json').open('w', encoding='utf-8') as file:
    json.dump(literature_case, file, indent=4)

fragility_parameters_df.to_csv(OUTPUT_DIR / 'published_fragility_parameters.csv', index=False)
fragility_points_df.to_csv(OUTPUT_DIR / 'published_fragility_points.csv', index=False)
design_speed_df.to_csv(OUTPUT_DIR / 'published_design_speed_probabilities.csv', index=False)

print(f'Saved notebook V3 outputs to: {OUTPUT_DIR}')